In [ ]:
%matplotlib inline

# Notebook 4 (optional) — Increase the IP beta functions

Try to increase both IP beta functions by a factor of five, from 0.09 m / 0.0008 m to 0.45 m / 0.0040 m. Keep a waist at the IP: $\alpha_x=\alpha_y=0$.

Match the two halves of the straight around `ipd`, keeping the nominal optics at the outer boundaries. Restore the nominal tunes and chromaticities with the arc circuits.

### Load the nominal lattice

In [ ]:
import xtrack as xt

env = xt.load('fccee_z.json')
env.vars.load('optics_nominal.json')
line = env.fccee_p_ring
line.configure_radiation(model=None)
tw0 = line.twiss4d(delta_chrom=1e-6)

### Inspect the experimental straight

In [ ]:
ip = 'ipd'
start = 'end_ds_start_straight_ipd'
end = 'end_straight_start_ds_ipd'

tw0.rows[[start, ip, end]].cols['s betx bety alfx alfy dx dpx']

In [ ]:
tw0.rows[start:end].plot()

### Quadrupoles for matching

Use the quadrupoles in the dispersion-free matching sections.

In [ ]:
knobs_left = ['kqd16l', 'kqf17l', 'kqd18l', 'kqf19l', 'kqd20l',
              'kqd8m', 'kqf9m', 'kqd10m']
knobs_right = ['kqf17r', 'kqd18r', 'kqf19r', 'kqd20r',
               'kqd8mr', 'kqf9mr', 'kqd10mr']

### Fix the boundary optics

Use the nominal Twiss parameters at the two outer boundaries as initial conditions for open matching.

In [ ]:
init_left = tw0.get_twiss_init(start)
init_right = tw0.get_twiss_init(end)

### Set the IP targets

Target five times the nominal beta functions and zero alpha in both planes. The supplied weights account for the different horizontal and vertical beta scales.

In [ ]:
beta_factor = 5

targets_ip = [
    xt.Target('betx', at=ip, value=..., tol=1e-8, weight=10),
    xt.Target('bety', at=ip, value=..., tol=1e-10, weight=1000),
    xt.TargetSet(alfx=..., alfy=..., at=ip, tol=1e-7),
]

### Match the left half

Propagate from `start` to `ip` using `init_left` and vary `knobs_left`. Create the optimizer with `solve=False`, then run the matching with `.solve()`.

In [ ]:
opt_left = line.match(
    solve=False,
    method='4d',
    start=..., end=..., init=...,
    vary=xt.VaryList(knobs_left, step=1e-8),
    targets=targets_ip,
    n_steps_max=80,
)

In [ ]:
opt_left.target_status()

In [ ]:
opt_left.solve()

In [ ]:
opt_left.target_status()

### Match the right half

Use the range from `ip` to `end`, with `init_right` specified at `end`. Set `init_at=end` to propagate backwards from the downstream boundary.

In [ ]:
opt_right = line.match(
    solve=False,
    method='4d',
    start=..., end=..., init=..., init_at=...,
    vary=xt.VaryList(knobs_right, step=1e-8),
    targets=targets_ip,
    n_steps_max=80,
)

In [ ]:
opt_right.target_status()

In [ ]:
opt_right.solve()

### Limit the horizontal beta peak

After matching the IP, add an `xt.LessThan(2000)` target on the maximum horizontal beta in the right half. Clone the optimizer to keep the same IP targets, quadrupoles and boundary optics, then solve again.

In [ ]:
opt_right = opt_right.clone(add_targets=[
    xt.Target(lambda tw: tw.betx.max(), xt.LessThan(2000),
              tol=1e-3, weight=1e-3, tag='max_betx'),
])

In [ ]:
opt_right.target_status()

In [ ]:
opt_right.solve()

In [ ]:
opt_right.target_status()

### Inspect the matched straight

Propagate through the entire straight from the fixed left boundary. Compare the IP beta functions and the recovered right-boundary optics with the nominal table above.

In [ ]:
tw_local = line.twiss4d(start=start, end=end, init=init_left)
tw_local.rows[[start, ip, end]].cols['s betx bety alfx alfy dx dpx']

In [ ]:
tw_local.plot()

### Restore the tunes

Use the arc quadrupole circuits `kqf2` and `kqd1` to recover the nominal tunes from `tw0`.

In [ ]:
opt_tunes = line.match(
    solve=False,
    method='4d',
    vary=xt.VaryList(['kqf2', 'kqd1'], step=1e-8),
    targets=xt.TargetSet(qx=..., qy=..., tol=1e-8),
)

In [ ]:
opt_tunes.target_status()

In [ ]:
opt_tunes.solve()

### Restore the chromaticities

Use the arc sextupole families `ksf1` and `ksd1` to recover the nominal chromaticities. Keep `delta_chrom=1e-6`, as in the nominal calculation.

In [ ]:
opt_chroma = line.match(
    solve=False,
    method='4d', delta_chrom=1e-6,
    vary=xt.VaryList(['ksf1', 'ksd1'], step=1e-5),
    targets=xt.TargetSet(dqx=..., dqy=..., tol=1e-3),
)

In [ ]:
opt_chroma.target_status()

In [ ]:
opt_chroma.solve()

### Check the periodic ring

Recalculate the periodic optics and check all four IPs, tunes, and chromaticities. The arc quadrupole adjustment can slightly change the IP beta functions.

In [ ]:
tw = line.twiss4d(delta_chrom=1e-6)
tw.rows[['ipa', 'ipd', 'ipg', 'ipj']].cols['betx bety alfx alfy']

In [ ]:
print('Nominal tunes:', tw0.qx, tw0.qy)
print('Larger IP beta tunes:', tw.qx, tw.qy)

In [ ]:
print('Nominal chromaticities:', tw0.dqx, tw0.dqy)
print('Larger IP beta chromaticities:', tw.dqx, tw.dqy)
# print('Maximum horizontal beta in right half [m]:', ...)